# 6.5 汇聚层

汇聚层用于降低卷积层对位置的敏感性，并逐步压缩空间尺寸。常见方式包括最大汇聚和平均汇聚。


In [1]:
import torch
from torch import nn


## 最大汇聚层和平均汇聚层


In [2]:
def pool2d(X, pool_size, mode='max'):
    p_h, p_w = pool_size
    Y = torch.zeros((X.shape[0] - p_h + 1, X.shape[1] - p_w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            patch = X[i:i + p_h, j:j + p_w]
            if mode == 'max':
                Y[i, j] = patch.max()
            elif mode == 'avg':
                Y[i, j] = patch.mean()
            else:
                raise ValueError('mode must be max or avg')
    return Y


X = torch.tensor([[0.0, 1.0, 2.0],
                  [3.0, 4.0, 5.0],
                  [6.0, 7.0, 8.0]])
print(pool2d(X, (2, 2)))
print(pool2d(X, (2, 2), 'avg'))


tensor([[4., 5.],
        [7., 8.]])
tensor([[2., 3.],
        [5., 6.]])


## 填充和步幅

PyTorch 的汇聚层同样支持填充、步幅和非方形窗口。


In [3]:
X = torch.arange(16, dtype=torch.float32).reshape((1, 1, 4, 4))

pool2d_layer = nn.MaxPool2d(3)
print(pool2d_layer(X))

pool2d_layer = nn.MaxPool2d(3, padding=1, stride=2)
print(pool2d_layer(X))

pool2d_layer = nn.MaxPool2d((2, 3), stride=(2, 3), padding=(0, 1))
print(pool2d_layer(X))


tensor([[[[10.]]]])
tensor([[[[ 5.,  7.],
          [13., 15.]]]])
tensor([[[[ 5.,  7.],
          [13., 15.]]]])


## 多个通道

汇聚层在每个输入通道上单独计算，不会像卷积层那样把通道相加。


In [4]:
X = torch.cat((X, X + 1), 1)
print(X.shape)

pool2d_layer = nn.MaxPool2d(3, padding=1, stride=2)
print(pool2d_layer(X))


torch.Size([1, 2, 4, 4])
tensor([[[[ 5.,  7.],
          [13., 15.]],

         [[ 6.,  8.],
          [14., 16.]]]])


最大汇聚更关注局部最强响应，平均汇聚更关注局部整体水平；现代 CNN 中最大汇聚更常用于提取显著特征。
